# GLiNER2 Smoke Test for Banking77 Intent Classification

This notebook verifies local GLiNER2 inference and a small zero-shot Banking77 evaluation in Google Colab. It does not fine-tune the model, create manual annotations, call the Pioneer / GLiNER2 cloud API, or implement the later schema-wording method.

In [ ]:
PROJECT_NAME = "gliner2_schema_wording"
MODEL_NAME = "fastino/gliner2-base-v1"
DATASET_NAME = "mteb/banking77"
MODE = "smoke"  # smoke / small_eval
USE_GPU_IF_AVAILABLE = True
SEED = 42
SMOKE_N_EXAMPLES = 10
SMALL_EVAL_PER_LABEL = 2
FORCE_RERUN = False
OUTPUT_DIR = "/content/gliner2_schema_outputs"


In [ ]:
from pathlib import Path
import json
import traceback

OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

RUN_STATE = {
    "installation_succeeded": "not run yet",
    "model_loading_succeeded": "not run yet",
    "device_used": "not run yet",
    "dataset_loaded": "not run yet",
    "num_evaluated_examples": "not run yet",
    "baseline_accuracy": "not run yet",
    "baseline_macro_f1": "not run yet",
    "baseline_weighted_f1": "not run yet",
    "average_latency": "not run yet",
    "gpu_memory": "not run yet",
    "observed_api_output_format": "not run yet",
    "failure_points": [],
    "safe_to_continue": "pending successful Colab execution",
}


def _format_state_value(key):
    value = RUN_STATE.get(key, "not run yet")
    if isinstance(value, float):
        return f"{value:.6f}"
    if isinstance(value, list):
        return "none" if not value else "; ".join(str(item) for item in value)
    return str(value)


def write_smoke_report(extra_targets=None):
    lines = [
        "# GLiNER2 Smoke Test Report",
        "",
        f"Status: {_format_state_value('safe_to_continue')}",
        "",
        "## Run Summary",
        "",
        f"- Installation succeeded: {_format_state_value('installation_succeeded')}",
        f"- Model loading succeeded: {_format_state_value('model_loading_succeeded')}",
        f"- Device used: {_format_state_value('device_used')}",
        f"- Dataset loaded: {_format_state_value('dataset_loaded')}",
        f"- Number of evaluated examples: {_format_state_value('num_evaluated_examples')}",
        f"- Baseline accuracy: {_format_state_value('baseline_accuracy')}",
        f"- Baseline macro F1: {_format_state_value('baseline_macro_f1')}",
        f"- Baseline weighted F1: {_format_state_value('baseline_weighted_f1')}",
        f"- Average latency: {_format_state_value('average_latency')}",
        f"- GPU memory: {_format_state_value('gpu_memory')}",
        f"- Observed API output format: {_format_state_value('observed_api_output_format')}",
        f"- Failure points: {_format_state_value('failure_points')}",
        f"- Safe to continue: {_format_state_value('safe_to_continue')}",
        "",
        "## Notes",
        "",
        f"- Dataset: `{DATASET_NAME}`",
        "- No fallback dataset is used if the primary dataset fails.",
        "- Baseline method: `single_schema_plain_label`",
        "- This report contains only values observed during this notebook run.",
    ]

    targets = [OUTPUT_PATH / "SMOKE_TEST_REPORT.md"]
    if "PROJECT_ROOT" in globals():
        targets.append(PROJECT_ROOT / "SMOKE_TEST_REPORT.md")
    if extra_targets:
        targets.extend(extra_targets)

    written = []
    for target in dict.fromkeys(Path(path) for path in targets):
        try:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_text("\n".join(lines) + "\n", encoding="utf-8")
            written.append(str(target))
        except Exception as exc:
            print(f"Could not write report to {target}: {exc}")
    print("Report written to:")
    for path in written:
        print(f"  {path}")


write_smoke_report()

In [ ]:
import subprocess
import sys
from pathlib import Path


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/GLiNER2-demo"),
        Path("/content/gliner2_schema_wording"),
        Path("/content/drive/MyDrive/GLiNER2-demo"),
        Path("/content/drive/MyDrive/gliner2_schema_wording"),
    ]
    for root in candidates:
        if (root / "requirements-colab.txt").exists() and (root / "src").exists():
            return root.resolve()

    for pattern in ["*/requirements-colab.txt", "*/*/requirements-colab.txt"]:
        for req_path in Path("/content").glob(pattern):
            root = req_path.parent
            if (root / "src").exists():
                return root.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    message = (
        "Could not find project root. Make sure requirements-colab.txt and src/ "
        "are available in the Colab runtime, then rerun this notebook."
    )
    RUN_STATE["failure_points"].append(message)
    write_smoke_report()
    raise FileNotFoundError(message)

print(f"Project root: {PROJECT_ROOT}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    RUN_STATE["installation_succeeded"] = "running"
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
    )
    RUN_STATE["installation_succeeded"] = "yes"
except Exception as exc:
    RUN_STATE["installation_succeeded"] = f"no: {exc}"
    RUN_STATE["failure_points"].append("Dependency installation failed: " + repr(exc))
    RUN_STATE["safe_to_continue"] = "no, dependency installation failed"
    write_smoke_report()
    raise

write_smoke_report()

In [ ]:
import inspect
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from gliner2_project.data_utils import (
    build_label_mappings,
    ensure_label_text_column,
    map_prediction_to_canonical,
    stratified_subset,
    unique_label_texts,
)
from gliner2_project.env_utils import print_environment_info
from gliner2_project.eval_utils import (
    average_latency,
    collect_confusion_examples,
    compute_classification_metrics,
)
from gliner2_project.gliner2_wrappers import (
    load_gliner2_model,
    make_json_safe,
    predict_intent,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Imports completed.")

In [ ]:
environment_info = print_environment_info()
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    RUN_STATE["gpu_memory"] = f"{total_gb:.2f} GB total"
else:
    RUN_STATE["gpu_memory"] = "not available"
write_smoke_report()

In [ ]:
if USE_GPU_IF_AVAILABLE and torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

RUN_STATE["device_used"] = DEVICE
if DEVICE == "cpu":
    print("WARNING: GPU is unavailable or disabled. CPU smoke inference on a few examples is allowed, but full evaluation may be slow.")
else:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

try:
    model = load_gliner2_model(MODEL_NAME, device=DEVICE)
    RUN_STATE["model_loading_succeeded"] = "yes"
    classify_text_method = getattr(model, "classify_text", None)
    if classify_text_method is None:
        raise AttributeError("Loaded model does not expose classify_text.")
    try:
        signature = inspect.signature(classify_text_method)
        print(f"classify_text signature: {signature}")
    except Exception as exc:
        print(f"Could not inspect classify_text signature: {exc}")
except Exception as exc:
    RUN_STATE["model_loading_succeeded"] = f"no: {exc}"
    RUN_STATE["failure_points"].append("Model loading failed: " + repr(exc))
    RUN_STATE["safe_to_continue"] = "no, model loading failed"
    write_smoke_report()
    raise

write_smoke_report()

In [ ]:
quick_examples = [
    "I still haven't received my new card after two weeks.",
    "I forgot my passcode and cannot access the app.",
    "Why was I charged twice for the same card transaction?",
    "Can I use Apple Pay with my card?",
    "I need to change my phone number.",
]
quick_candidate_labels = [
    "card arrival",
    "passcode forgotten",
    "cash withdrawal",
    "apple pay or google pay",
    "edit personal details",
    "transaction charged twice",
]

quick_results = []
for text in quick_examples:
    prediction = predict_intent(model, text, quick_candidate_labels, task_name="intent")
    raw_output = make_json_safe(prediction.raw_output)
    quick_results.append(raw_output)
    if RUN_STATE["observed_api_output_format"] == "not run yet":
        RUN_STATE["observed_api_output_format"] = f"{type(prediction.raw_output).__name__}: {repr(raw_output)[:500]}"

    print("=" * 80)
    print(f"input text: {text}")
    print(f"candidate labels: {quick_candidate_labels}")
    print("raw GLiNER2 output:")
    print(json.dumps(raw_output, ensure_ascii=False, indent=2)[:2000])
    print(f"parsed predicted label: {prediction.label}")
    print(f"confidence: {prediction.confidence}")

write_smoke_report()

In [ ]:
from datasets import load_dataset

try:
    dataset = load_dataset(DATASET_NAME)
    RUN_STATE["dataset_loaded"] = "yes"
except Exception as exc:
    RUN_STATE["dataset_loaded"] = f"no: {exc}"
    RUN_STATE["failure_points"].append(f"Dataset loading failed for {DATASET_NAME}: {repr(exc)}")
    RUN_STATE["safe_to_continue"] = "no, primary dataset failed to load"
    write_smoke_report()
    raise RuntimeError(
        f"Failed to load primary dataset {DATASET_NAME}. No fallback dataset is used."
    ) from exc

print(f"Split names: {list(dataset.keys())}")
for split_name, split in dataset.items():
    print(f"{split_name}: {len(split)} examples")
    print(f"{split_name} columns: {split.column_names}")

if "test" not in dataset:
    message = "Dataset does not contain a test split."
    RUN_STATE["failure_points"].append(message)
    RUN_STATE["safe_to_continue"] = "no, dataset validation failed"
    write_smoke_report()
    raise ValueError(message)

test_split = dataset["test"]
if "text" not in test_split.column_names:
    message = "Test split does not contain a text column."
    RUN_STATE["failure_points"].append(message)
    RUN_STATE["safe_to_continue"] = "no, dataset validation failed"
    write_smoke_report()
    raise ValueError(message)

if "label_text" not in test_split.column_names:
    print("label_text is missing. Attempting to recover label names from dataset features.")
    try:
        test_split = ensure_label_text_column(test_split)
    except Exception as exc:
        RUN_STATE["failure_points"].append("label_text recovery failed: " + repr(exc))
        RUN_STATE["safe_to_continue"] = "no, label_text recovery failed"
        write_smoke_report()
        raise

print(f"Validated test columns: {test_split.column_names}")
print(test_split.select(range(min(3, len(test_split)))).to_pandas())
write_smoke_report()

In [ ]:
label_texts = unique_label_texts(test_split)
canonical_to_display, display_to_canonical = build_label_mappings(label_texts)
candidate_labels = [canonical_to_display[label] for label in label_texts]

print(f"Number of candidate labels: {len(candidate_labels)}")
if len(candidate_labels) != 77:
    print(f"WARNING: Expected 77 Banking77 labels, observed {len(candidate_labels)} labels.")
print("First 10 candidate labels:", candidate_labels[:10])

stratified_rows = stratified_subset(
    test_split,
    per_label=SMALL_EVAL_PER_LABEL,
    seed=SEED,
)
print(f"Stratified subset size before mode cap: {len(stratified_rows)}")

if MODE == "smoke":
    eval_examples = stratified_rows[:SMOKE_N_EXAMPLES]
elif MODE == "small_eval":
    eval_examples = stratified_rows
else:
    raise ValueError("MODE must be either 'smoke' or 'small_eval'.")

if DEVICE == "cpu" and MODE == "smoke" and len(eval_examples) > 5:
    print("CPU mode detected; capping smoke evaluation to 5 examples.")
    eval_examples = eval_examples[:5]

print(f"Evaluation mode: {MODE}")
print(f"Number of examples selected: {len(eval_examples)}")

In [ ]:
METHOD_NAME = "single_schema_plain_label"
predictions_dir = Path(OUTPUT_DIR) / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)
predictions_path = predictions_dir / "smoke_predictions.jsonl"

if predictions_path.exists() and not FORCE_RERUN:
    print(f"Loading existing predictions from {predictions_path}. Set FORCE_RERUN=True to recompute.")
    prediction_rows = [
        json.loads(line)
        for line in predictions_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    prediction_rows = []
    for example in tqdm(eval_examples, desc="GLiNER2 baseline"):
        start_time = time.perf_counter()
        prediction = predict_intent(model, example["text"], candidate_labels, task_name="intent")
        latency_sec = time.perf_counter() - start_time

        predicted_display_label = prediction.label
        predicted_canonical_label = map_prediction_to_canonical(
            predicted_display_label,
            display_to_canonical,
        )

        row = {
            "example_id": example.get("example_id"),
            "text": example["text"],
            "gold_label": example["label_text"],
            "candidate_labels": candidate_labels,
            "raw_output": make_json_safe(prediction.raw_output),
            "predicted_display_label": predicted_display_label,
            "predicted_canonical_label": predicted_canonical_label,
            "confidence": prediction.confidence,
            "latency_sec": latency_sec,
            "method": METHOD_NAME,
        }
        prediction_rows.append(row)

    with predictions_path.open("w", encoding="utf-8") as f:
        for row in prediction_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Saved predictions to {predictions_path}")

RUN_STATE["num_evaluated_examples"] = len(prediction_rows)
if prediction_rows and RUN_STATE["observed_api_output_format"] == "not run yet":
    RUN_STATE["observed_api_output_format"] = (
        f"{type(prediction_rows[0]['raw_output']).__name__}: "
        f"{repr(prediction_rows[0]['raw_output'])[:500]}"
    )
write_smoke_report()

In [ ]:
gold_labels = [row["gold_label"] for row in prediction_rows]
predicted_labels = [row.get("predicted_canonical_label") for row in prediction_rows]
metrics = compute_classification_metrics(gold_labels, predicted_labels)
latencies = [float(row.get("latency_sec", 0.0)) for row in prediction_rows]
avg_latency = average_latency(latencies)

confusion_examples = collect_confusion_examples(
    [
        {
            "example_id": row.get("example_id"),
            "text": row.get("text"),
            "label_text": row.get("gold_label"),
        }
        for row in prediction_rows
    ],
    predicted_labels,
    max_examples=10,
)

RUN_STATE["baseline_accuracy"] = metrics["accuracy"]
RUN_STATE["baseline_macro_f1"] = metrics["macro_f1"]
RUN_STATE["baseline_weighted_f1"] = metrics["weighted_f1"]
RUN_STATE["average_latency"] = avg_latency
RUN_STATE["safe_to_continue"] = "yes, if the numbers above are acceptable for a smoke test"

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    allocated_gb = torch.cuda.memory_allocated(0) / 1024**3
    reserved_gb = torch.cuda.memory_reserved(0) / 1024**3
    RUN_STATE["gpu_memory"] = (
        f"{total_gb:.2f} GB total, {allocated_gb:.2f} GB allocated, {reserved_gb:.2f} GB reserved"
    )

print("Metrics:")
print(json.dumps(metrics, indent=2))
print(f"Average latency per example: {avg_latency:.3f} sec")
print("Confusion examples:")
print(json.dumps(confusion_examples, ensure_ascii=False, indent=2)[:4000])

write_smoke_report()
print("Smoke test complete.")